In [1]:
platformID = 'FBE'

## import libraries

In [2]:
import os
import zipfile

from tqdm import tqdm 
from datetime import datetime

import pandas as pd
pd.set_option('display.max_colwidth', None)

import numpy as np

import re

import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns 

import psycopg2

## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

from functions import lookup_loader
import test_functions

In [4]:
lookup = lookup_loader(gam_info, platformID, '3',
                       with_country=True)
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']


✅ Test FBE_3_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_3_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_3_02 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_3_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_3_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_3_05 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_3_06 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_3_07 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_3_08 passed: No missing values in lookup.
...updating logbook...



# Unique Viewers

In [5]:
facebook_engagements_reach = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT.csv",)
facebook_engagements_reach['w/c'] = pd.to_datetime(facebook_engagements_reach['w/c'])



In [6]:
facebook_engagements_reach.sort_values(['w/c'])['w/c'].unique()

<DatetimeArray>
['2025-03-31 00:00:00', '2025-04-07 00:00:00', '2025-04-14 00:00:00',
 '2025-04-21 00:00:00', '2025-04-28 00:00:00', '2025-05-05 00:00:00',
 '2025-05-12 00:00:00', '2025-05-19 00:00:00', '2025-05-26 00:00:00',
 '2025-06-02 00:00:00', '2025-06-09 00:00:00', '2025-06-16 00:00:00',
 '2025-06-23 00:00:00', '2025-06-30 00:00:00', '2025-07-07 00:00:00',
 '2025-07-14 00:00:00', '2025-07-21 00:00:00', '2025-07-28 00:00:00',
 '2025-08-04 00:00:00', '2025-08-11 00:00:00', '2025-08-18 00:00:00',
 '2025-08-25 00:00:00', '2025-09-01 00:00:00', '2025-09-08 00:00:00',
 '2025-09-15 00:00:00', '2025-09-22 00:00:00', '2025-09-29 00:00:00',
 '2025-10-06 00:00:00', '2025-10-13 00:00:00', '2025-10-20 00:00:00',
 '2025-10-27 00:00:00', '2025-11-03 00:00:00', '2025-11-10 00:00:00',
 '2025-11-17 00:00:00', '2025-11-24 00:00:00', '2025-12-01 00:00:00',
 '2025-12-08 00:00:00']
Length: 37, dtype: datetime64[ns]

# Country

In [7]:
country_raw = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_COUNTRY.csv")
country_raw['w/c'] = pd.to_datetime(country_raw['w/c'])

cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
country_df = country_raw[cols]
country_df['PlatformID'] = platformID
country_df.head()

,Channel ID,Channel Name,w/c,PlaceID,country_%,PlatformID
0,FBE237388053065908,BBC Culture,2025-03-31,MAL,0.013062,FBE
1,FBE237388053065908,BBC Culture,2025-03-31,MEX,0.025156,FBE
2,FBE237388053065908,BBC Culture,2025-03-31,POL,0.010158,FBE
3,FBE237388053065908,BBC Culture,2025-03-31,AUS,0.022314,FBE
4,FBE237388053065908,BBC Culture,2025-03-31,SAF,0.006610,FBE


In [8]:
country_df.sort_values(['w/c'])['w/c'].unique()

<DatetimeArray>
['2025-03-31 00:00:00', '2025-04-07 00:00:00', '2025-04-14 00:00:00',
 '2025-04-21 00:00:00', '2025-04-28 00:00:00', '2025-05-05 00:00:00',
 '2025-05-12 00:00:00', '2025-05-19 00:00:00', '2025-05-26 00:00:00',
 '2025-06-02 00:00:00', '2025-06-09 00:00:00', '2025-06-16 00:00:00',
 '2025-06-23 00:00:00', '2025-06-30 00:00:00', '2025-07-07 00:00:00',
 '2025-07-14 00:00:00', '2025-07-21 00:00:00', '2025-07-28 00:00:00',
 '2025-08-04 00:00:00', '2025-08-11 00:00:00', '2025-08-18 00:00:00',
 '2025-08-25 00:00:00', '2025-09-01 00:00:00', '2025-09-08 00:00:00',
 '2025-09-15 00:00:00', '2025-09-22 00:00:00', '2025-09-29 00:00:00',
 '2025-10-06 00:00:00', '2025-10-13 00:00:00', '2025-10-20 00:00:00',
 '2025-10-27 00:00:00', '2025-11-03 00:00:00', '2025-11-10 00:00:00',
 '2025-11-17 00:00:00', '2025-11-24 00:00:00', '2025-12-01 00:00:00',
 '2025-12-08 00:00:00', '2025-12-15 00:00:00', '2025-12-22 00:00:00']
Length: 39, dtype: datetime64[ns]

# combine engagements & country

In [9]:
country_df[(country_df['Channel ID'] == 'FBE630866223444617') 
    #& (country_df['w/c'].isin(['2025-06-23', '2025-06-30']))
    ]

,Channel ID,Channel Name,w/c,PlaceID,country_%,PlatformID
176705,FBE630866223444617,NaN,2025-06-23,ALG,0.004225,FBE
176706,FBE630866223444617,NaN,2025-06-23,ANG,0.000542,FBE
176707,FBE630866223444617,NaN,2025-06-23,ARG,0.000262,FBE
176708,FBE630866223444617,NaN,2025-06-23,AUS,0.002099,FBE
176709,FBE630866223444617,NaN,2025-06-23,BAN,0.000262,FBE
...,...,...,...,...,...,...
178091,FBE630866223444617,NaN,2025-12-22,UAE,0.000281,FBE
178092,FBE630866223444617,NaN,2025-12-22,UK,0.118736,FBE
178093,FBE630866223444617,NaN,2025-12-22,UKR,0.001748,FBE
178094,FBE630866223444617,NaN,2025-12-22,USA,0.016809,FBE


In [10]:
reach_df_raw = facebook_engagements_reach.merge(country_df, on=['Channel ID', 'w/c'], how='outer', 
                                            indicator=True)
print(reach_df_raw._merge.value_counts())
reach_df_left = reach_df_raw[reach_df_raw['_merge'] == 'left_only'].drop(columns=['_merge'])
reach_df = reach_df_raw[reach_df_raw['_merge'] == 'both'].drop(columns=['_merge'])

'''
reach_df_inner = reach_df_raw[reach_df_raw['_merge'] == 'both'].drop(columns=['_merge'])
reach_df_avg = reach_df_left[facebook_engagements_reach.columns].merge(avg_country_df, 
                                    on=['Channel ID', 'w/c'], how='left', indicator=True)

reach_df = pd.concat([reach_df_inner, reach_df_avg])'''

_merge
both          168151
right_only      9945
left_only          0
Name: count, dtype: int64


"\nreach_df_inner = reach_df_raw[reach_df_raw['_merge'] == 'both'].drop(columns=['_merge'])\nreach_df_avg = reach_df_left[facebook_engagements_reach.columns].merge(avg_country_df, \n                                    on=['Channel ID', 'w/c'], how='left', indicator=True)\n\nreach_df = pd.concat([reach_df_inner, reach_df_avg])"

In [11]:
metric_col = ['country_%', 'engaged_reach']
for col in metric_col:
    reach_df[col] = reach_df[col].fillna(0)
    
reach_df['uv_by_country'] = reach_df['country_%'] * reach_df['engaged_reach']
reach_df = reach_df[reach_df['uv_by_country'] > 0]
# TODO investigate why there should be duplicates here
reach_df = reach_df.drop_duplicates()

print(reach_df.shape)
reach_df = reach_df.dropna(subset='uv_by_country')
print(reach_df.shape)

(167238, 9)
(167238, 9)


In [12]:
# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col='Channel ID',
                                               main_data=reach_df,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts,
                                               test_number=f"{platformID}_3_09",
                                               test_step="Check all weeks present for each account")


❌ Missing weeks from Start onward:
          Start        End          Channel ID        w/c
195  2020-04-01        NaT  FBE102775241582303 2025-03-31
196  2020-04-01        NaT  FBE102775241582303 2025-04-07
197  2020-04-01        NaT  FBE102775241582303 2025-04-14
199  2020-04-01        NaT  FBE102775241582303 2025-04-28
1602 2025-04-02 2025-08-19  FBE171824429536304 2025-04-28
...         ...        ...                 ...        ...
857  2020-04-01        NaT  FBE127616880628817 2025-12-22
2104 2020-04-01        NaT     FBE215963631764 2025-12-22
2065 2020-04-01        NaT  FBE207150596007088 2025-12-22
2650 2020-04-01        NaT  FBE303095973088848 2025-12-22
3925 2020-04-01        NaT  FBE948946275170651 2025-12-22

[227 rows x 4 columns]

Summary of missing weeks per channel_id_col:
              Channel ID  missing_week_count
5     FBE102775241582303                  25
21    FBE127284984048667                   3
43    FBE171824429536304                   3
0     FBE1001639901

,Start,End,Channel ID,w/c
195,2020-04-01,NaT,FBE102775241582303,2025-03-31
196,2020-04-01,NaT,FBE102775241582303,2025-04-07
197,2020-04-01,NaT,FBE102775241582303,2025-04-14
199,2020-04-01,NaT,FBE102775241582303,2025-04-28
1602,2025-04-02,2025-08-19,FBE171824429536304,2025-04-28
...,...,...,...,...
857,2020-04-01,NaT,FBE127616880628817,2025-12-22
2104,2020-04-01,NaT,FBE215963631764,2025-12-22
2065,2020-04-01,NaT,FBE207150596007088,2025-12-22
2650,2020-04-01,NaT,FBE303095973088848,2025-12-22


In [13]:
reach_df.sort_values(['w/c'])['w/c'].unique()

<DatetimeArray>
['2025-03-31 00:00:00', '2025-04-07 00:00:00', '2025-04-14 00:00:00',
 '2025-04-21 00:00:00', '2025-04-28 00:00:00', '2025-05-05 00:00:00',
 '2025-05-12 00:00:00', '2025-05-19 00:00:00', '2025-05-26 00:00:00',
 '2025-06-02 00:00:00', '2025-06-09 00:00:00', '2025-06-16 00:00:00',
 '2025-06-23 00:00:00', '2025-06-30 00:00:00', '2025-07-07 00:00:00',
 '2025-07-14 00:00:00', '2025-07-21 00:00:00', '2025-07-28 00:00:00',
 '2025-08-04 00:00:00', '2025-08-11 00:00:00', '2025-08-18 00:00:00',
 '2025-08-25 00:00:00', '2025-09-01 00:00:00', '2025-09-08 00:00:00',
 '2025-09-15 00:00:00', '2025-09-22 00:00:00', '2025-09-29 00:00:00',
 '2025-10-06 00:00:00', '2025-10-13 00:00:00', '2025-10-20 00:00:00',
 '2025-10-27 00:00:00', '2025-11-03 00:00:00', '2025-11-10 00:00:00',
 '2025-11-17 00:00:00', '2025-11-24 00:00:00', '2025-12-01 00:00:00',
 '2025-12-08 00:00:00']
Length: 37, dtype: datetime64[ns]

In [14]:
cols = ['ServiceID', 'Channel ID', 'w/c', 'PlaceID', 'uv_by_country']
reach_df[cols].to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_uniqueViewer_country.csv", 
                     index=None)